# Auditing components.yaml

This notebook runs the AuditSystem against the main components.yaml file to evaluate solution quality.

In [ ]:
from pathlib import Path

import ibis
from hamilton import driver, base

from iacs.audit_system import AuditRunner
from iacs.transforms import manifest_to_registry
from iacs.transform_system import TransformSystem

ibis.options.interactive = True  # auto-display results (like pandas)

## Load components.yaml

In [2]:
COMPONENTS_DIR = str(Path("../components"))

# Build registry from YAML manifests using Hamilton DAG
build_driver = driver.Driver(
    {"input_dir": COMPONENTS_DIR},
    manifest_to_registry,
    adapter=base.DictResult(),
)
result = build_driver.execute(["registry"])
reg = result["registry"]

# Create TransformSystem for post-registry transforms
ts = TransformSystem(reg, [manifest_to_registry])

print(f"Component types: {reg.component_types}")
print(f"Number of component types: {len(reg.component_types)}")
print(f"Available transforms: {ts.outputs}")

Component types: ['name', 'description', 'file_info', 'requirement', 'parent', 'id', 'todo', 'system', 'input', 'output', 'weight']
Number of component types: 11
Available transforms: ['complete_schema', 'component_first_data', 'schema', 'parent', 'components_database', 'conn', 'components', 'data_models', 'flattened_entity_first_data', 'flattened_data', 'name_to_id', 'raw_entity_first_data', 'validated_components']


## Run All Audits

In [ ]:
runner = AuditRunner.default()

results = runner.run(reg)

print(f"All audits passed: {runner.all_passed}")
print()
for name, table in results.items():
    count = table.count().execute()
    status = "PASS" if count == 0 else "FAIL"
    print(f"{name}: {status} ({count} issues)")

## Requirement Coverage Audit

Checks that all requirements have implementing solutions.

In [ ]:
results["requirement_coverage"]

## Traceability Audit

Checks that all entities trace back to requirements.

In [ ]:
results["traceability"]

## Todo Audit

Reports outstanding todos.

In [ ]:
results["todo"]